# ExoSignal Deep Learning Optimization

GPU notebook for improving the CNN beyond the first baseline. Run with a T4 GPU runtime. The notebook keeps binary results comparable to XGBoost first, then runs three-class experiments.

In [ ]:
# 1) Runtime check
import os, sys, json, platform
print('Python', sys.version)
print('Platform', platform.platform())
try:
    import tensorflow as tf
    print('TensorFlow before install', tf.__version__)
    print('GPUs', tf.config.list_physical_devices('GPU'))
except Exception as exc:
    print('TensorFlow not ready yet:', exc)

In [ ]:
# 2) Clone or update the project repo
REPO_URL = 'https://github.com/Maaskk/exosignal-observatory.git'
PROJECT_DIR = '/content/exosignal-observatory'

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    %cd {PROJECT_DIR}
    !git pull --ff-only
%cd {PROJECT_DIR}
!pwd
!ls -la | sed -n '1,80p'

In [ ]:
# 3) Install GPU training dependencies
!pip -q install -r requirements-deep.txt

import tensorflow as tf
print('TensorFlow', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))
if not tf.config.list_physical_devices('GPU'):
    raise RuntimeError('No GPU detected. In Colab: Runtime -> Change runtime type -> T4 GPU')

In [ ]:
# 4) Download/check full cleaned dataset
!python scripts/download_dataset.py --splits metadata train val test
!du -sh data data/train.parquet data/val.parquet data/test.parquet data/metadata.csv

In [ ]:
# 5) Memory investigation for longer sequences
!python scripts/train_deep_learning.py --print-memory-plan --sweep-sequence-lengths 512,1024,2048

In [ ]:
# 6) Smoke test: validates tensors/model/callbacks quickly
!python scripts/train_deep_learning.py \
  --smoke \
  --task binary \
  --model-family hybrid \
  --sequence-length 512 \
  --max-train-rows 512 \
  --max-val-rows 256 \
  --max-test-rows 256 \
  --mixed-precision \
  --augment \
  --loss focal

In [ ]:
# 7) Binary sequence-length sweep: apples-to-apples against XGBoost
!python scripts/train_deep_learning.py \
  --task binary \
  --model-family hybrid \
  --sweep-sequence-lengths 512,1024,2048 \
  --epochs 35 \
  --batch-size 32 \
  --filters 64 \
  --kernel-size 7 \
  --dropout 0.3 \
  --mixed-precision \
  --augment \
  --loss focal \
  --lr-schedule cosine

In [ ]:
# 8) KerasTuner Random Search on the strongest practical default: hybrid + 1024
!python scripts/train_deep_learning.py \
  --task binary \
  --model-family hybrid \
  --sequence-length 1024 \
  --epochs 30 \
  --tune \
  --tuner random \
  --max-trials 24 \
  --mixed-precision \
  --augment \
  --loss focal \
  --lr-schedule cosine

In [ ]:
# 9) Alternative architecture: TCN for long-range time dependencies
!python scripts/train_deep_learning.py \
  --task binary \
  --model-family tcn \
  --sequence-length 1024 \
  --epochs 35 \
  --batch-size 32 \
  --filters 64 \
  --kernel-size 7 \
  --dropout 0.3 \
  --mixed-precision \
  --augment \
  --loss focal \
  --lr-schedule cosine

In [ ]:
# 10) Three-class experiment: PLANET vs FALSE_POSITIVE vs NO_SIGNAL
!python scripts/train_deep_learning.py \
  --task multiclass \
  --model-family attention \
  --sequence-length 1024 \
  --epochs 40 \
  --batch-size 32 \
  --filters 64 \
  --kernel-size 7 \
  --dropout 0.3 \
  --mixed-precision \
  --augment \
  --loss focal \
  --lr-schedule cosine

In [ ]:
# 11) Show final reports
import json
from pathlib import Path
for path in [Path('models/deep_model_metrics.json'), Path('models/deep_model_config.json'), Path('reports/deep_learning_experiment_results.json'), Path('reports/deep_learning_tuner_result.json')]:
    print('\n====', path, '====')
    if path.exists():
        print(path.read_text()[:5000])
    else:
        print('missing')

In [ ]:
# 12) Export trained artifacts
from pathlib import Path
import zipfile
from google.colab import files

zip_path = Path('/content/exosignal_deep_optimized_artifacts.zip')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for pattern in ['models/*.json', 'models/*.keras', 'reports/*.json', 'reports/*.md']:
        for file in Path('.').glob(pattern):
            zf.write(file, file.as_posix())
print('artifact zip', zip_path, round(zip_path.stat().st_size / 1024 / 1024, 2), 'MB')
files.download(str(zip_path))